# Minakshi Polymers – handwritten form extraction with Qwen3-VL-4B

Pipeline for the scanned QA registers (Supplier Rejection / Rework / Segregation / Lot Rejection Summary, Weld Shop Line Rejection, Scrap Note):

```
PDF ─► render (applies PDF rotation) ─► normalise width ─► deskew ─► flatten lighting
    ─► detect ruled lines ─► drop fake lines (shadow edges) ─► find handwritten rows
    ─► group wrapped lines into records ─► crop [column header + record]
    ─► Qwen3-VL-4B (JSON) ─► validate (arithmetic, formats, ditto marks) ─► CSV / Excel / HTML report
```

**Input and output live on Google Drive** (same layout as the GOT v5.2 notebook):
PDFs are read from `MyDrive/MeenakshiPublic/Datasetpdf/`, and everything is written to
`MyDrive/MeenakshiPublic/qwen3vl_output/` - nothing is lost when the Colab runtime ends.

**Everything sent to the model is saved**, so you can see exactly what Qwen looked at:

```
MeenakshiPublic/qwen3vl_output/
  <FORM_FILE>/
    debug/          00_rendered … 07_grid_overlay.png   ← every preprocessing step
    model_inputs/   title.png, record_00.png …          ← the exact files passed to Qwen
    model_view/     record_00.png …                     ← same image at the resolution Qwen actually used
    model_io.jsonl                                      ← prompt, image size, tokens, raw output, parsed JSON, time
  results.csv / results.xlsx                            ← extracted records + validation status
  report.html                                           ← side-by-side: input image | output | issues
  truth_template.json                                   ← copy → correct → save as ground_truth.json to score accuracy
```

**Runtime:** Colab *T4 GPU* is enough (Qwen3-VL-4B in fp16 ≈ 9 GB). `Runtime → Change runtime type → T4 GPU`.
Set `DRY_RUN = True` to run only the preprocessing (no GPU) and inspect the crops first.

## 1. Install & imports

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Qwen3-VL needs transformers >= 4.57
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers>=4.57.0", "accelerate", "pymupdf", "opencv-python-headless", "openpyxl"],
                   check=True)
print("Running in Colab:", IN_COLAB)

In [ ]:
import os, re, json, time, shutil, html
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
import pymupdf
import matplotlib.pyplot as plt

try:
    from IPython.display import display, HTML
except ImportError:                       # plain python (local testing)
    display, HTML = print, str

try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
    print("torch", torch.__version__, "| CUDA:", HAS_CUDA)
    if HAS_CUDA:
        p = torch.cuda.get_device_properties(0)
        print(f"GPU: {p.name}  {p.total_memory / 1e9:.1f} GB")
except ImportError:
    torch, HAS_CUDA = None, False
    print("torch not installed")
print("opencv", cv2.__version__, "| numpy", np.__version__, "| pymupdf", pymupdf.__version__)

## 2. Configuration

In [ ]:
# ---- permanent storage (Colab): input AND output on Google Drive -----------------
# Colab's own disk (/content) is wiped when the runtime ends, so on Colab the PDFs are read from Drive and every
# output file is written to Drive. Locally (outside Colab) the two local folders below are used instead.
DRIVE_ROOT = "/content/drive/MyDrive/MeenakshiPublic"

def mount_drive():
    # Mount Google Drive once (Colab only). Safe to call again - a no-op when already mounted.
    if not IN_COLAB:
        return
    from google.colab import drive                      # only available inside Colab
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")

mount_drive()                                           # needed before anything below touches DRIVE_ROOT

INPUT_DIR  = Path(f"{DRIVE_ROOT}/Datasetpdf")     if IN_COLAB else Path("Minakshi Polymers")   # the PDFs
OUTPUT_DIR = Path(f"{DRIVE_ROOT}/qwen3vl_output") if IN_COLAB else Path("qwen_outputs")        # everything written
ONLY_FILES = []                                         # e.g. ["WELD", "SCRAP"] to process only matching file names

MODEL_ID       = "Qwen/Qwen3-VL-4B-Instruct"
DRY_RUN        = False    # True = preprocessing + saved model inputs only, no model call
MAX_NEW_TOKENS = 1024

# preprocessing (pixel values are at TARGET_WIDTH, i.e. an A4 page at ~300 dpi)
TARGET_WIDTH         = 2480
MIN_ROW_INK          = 1500   # handwriting pixels needed for a ruled row to count as filled
CONTINUATION_MAX_INK = 150    # if the first column (Date / S.No) has less ink, the row continues the previous entry
TOP_PAD              = 10     # extra px above a record (only when the row above is empty)
BOTTOM_PAD           = 22     # extra px below: people write ON the ruled line, descenders cross it
SIDE_PAD             = 40     # extra px left/right: circled S.No and long remarks run past the table border
IMAGE_MARGIN         = 20     # white border added around every model input

if not DRY_RUN and not HAS_CUDA:
    print("⚠ No GPU found -> switching to DRY_RUN (preprocessing only). "
          "In Colab: Runtime → Change runtime type → T4 GPU, then re-run.")
    DRY_RUN = True

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"input folder not found: {INPUT_DIR} - put the PDFs in "
                            f"{'Google Drive → MyDrive/MeenakshiPublic/Datasetpdf' if IN_COLAB else INPUT_DIR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("input :", INPUT_DIR.resolve())
print("output:", OUTPUT_DIR.resolve())
print("DRY_RUN:", DRY_RUN)

### PDFs to process
Every `.pdf` in `INPUT_DIR` (Drive: `MyDrive/MeenakshiPublic/Datasetpdf`). Add files there in Drive and re-run.

In [ ]:
PDFS = sorted(p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".pdf")
if ONLY_FILES:
    PDFS = [p for p in PDFS if any(s.upper() in p.name.upper() for s in ONLY_FILES)]
if not PDFS:
    print(f"⚠ no PDFs in {INPUT_DIR}")
print(f"{len(PDFS)} PDF(s):")
for p in PDFS:
    print("  ", p.name)

## 3. Form templates
One entry per form layout. The template holds **structure, not pixel coordinates**:
how many printed rows sit above the data (`header_rows`), how many vertical lines the table has
(`expected_col_lines`, used as a sanity check), the fields to extract (key → printed column header, type)
and the arithmetic/format rules used for validation.

In [ ]:
def S(header): return (header, "string")
def I(header): return (header, "integer")

TEMPLATES = {
    "SUPPLIER_LOT_REJECTION_SUMMARY": dict(
        match=["LOT REJECTION"], title="Supplier Lot Rejection Summary",
        header_rows=3, footer_rows=1, expected_col_lines=13, first_col="date",
        fields={"date": S("DATE"), "part_name": S("PART NAME"), "supplier_name": S("SUPPLIER NAME"),
                "lot_qty": I("LOT QTY"), "rework_qty": I("REW QTY"), "rejection_qty": I("REJ QTY"),
                "ok_qty": I("OK QTY"), "problem_description": S("PROBLEM DESCRIPTION"), "action": S("ACTION"),
                "return_status": S("RETURN STATUS"), "capa_status": S("CAPA STATUS"), "remarks": S("REMARKS")},
        required=["date", "part_name", "lot_qty", "ok_qty"],
        rules=[("lot_qty == rejection_qty + ok_qty", ["lot_qty", "rejection_qty", "ok_qty"],
                lambda r: r["lot_qty"] == r["rejection_qty"] + r["ok_qty"])]),
    "SUPPLIER_REJECTION": dict(
        match=["SUPPLIER REJECTION"], title="Supplier Rejection Report",
        header_rows=3, footer_rows=1, expected_col_lines=9, first_col="date",
        fields={"date": S("DATE"), "part_name": S("PART NAME"), "supplier_name": S("SUPPLIER NAME"),
                "lot_qty": I("LOT QTY"), "rejection_qty": I("REJECTION QTY"), "ok_qty": I("OK QTY"),
                "problem_description": S("PROBLEM DESCRIPTION"), "remarks": S("REMARKS")},
        required=["date", "part_name", "lot_qty", "rejection_qty", "ok_qty"],
        rules=[("lot_qty == rejection_qty + ok_qty", ["lot_qty", "rejection_qty", "ok_qty"],
                lambda r: r["lot_qty"] == r["rejection_qty"] + r["ok_qty"])]),
    "SUPPLIER_REWORK": dict(
        match=["REWORK"], title="Supplier Rework Report",
        header_rows=3, footer_rows=1, expected_col_lines=9, first_col="date",
        fields={"date": S("DATE"), "part_name": S("PART NAME"), "supplier_name": S("SUPPLIER NAME"),
                "lot_qty": I("LOT QTY"), "rework_qty": I("REWORK QTY"), "ok_qty": I("OK QTY"),
                "problem_description": S("PROBLEM DESCRIPTION"), "remarks": S("REMARKS")},
        required=["date", "part_name", "lot_qty", "rework_qty", "ok_qty"],
        rules=[("rework_qty <= lot_qty", ["rework_qty", "lot_qty"], lambda r: r["rework_qty"] <= r["lot_qty"]),
               ("ok_qty <= lot_qty", ["ok_qty", "lot_qty"], lambda r: r["ok_qty"] <= r["lot_qty"])]),
    "SUPPLIER_SEGREGATION": dict(
        match=["SEGREGATION"], title="Supplier Segregation Report",
        header_rows=3, footer_rows=1, expected_col_lines=9, first_col="date",
        fields={"date": S("DATE"), "part_name": S("PART NAME"), "supplier_name": S("SUPPLIER NAME"),
                "lot_qty": I("LOT QTY"), "segregation_qty": I("SEGREGATION QTY"), "ok_qty": I("OK QTY"),
                "problem_description": S("PROBLEM DESCRIPTION"), "remarks": S("REMARKS")},
        required=["date", "part_name", "lot_qty", "segregation_qty", "ok_qty"],
        rules=[("lot_qty == segregation_qty + ok_qty", ["lot_qty", "segregation_qty", "ok_qty"],
                lambda r: r["lot_qty"] == r["segregation_qty"] + r["ok_qty"])]),
    "WELD_SHOP_LINE_REJECTION": dict(
        match=["WELD SHOP"], title="Line Rejection Report – Weld Shop",
        header_rows=5, footer_rows=1, expected_col_lines=8, first_col="s_no",
        fields={"s_no": I("S.NO."), "item_code": S("ITEM CODE"), "material_particulars": S("MATERIAL PARTICULARS"),
                "unit": S("UNIT"), "rejection_qty": I("REJECTION QTY"), "vendor_name": S("VENDOR NAME"),
                "reason_remarks": S("REASON / REMARKS")},
        required=["item_code", "rejection_qty"],
        rules=[("item_code looks like SF500257 / RM500003", ["item_code"],
                lambda r: re.fullmatch(r"[A-Z]{2}\d{6}", r["item_code"].replace(" ", "").upper()) is not None)]),
    "SCRAP_NOTE": dict(
        match=["SCRAP"], title="Scrap Note",
        header_rows=4, footer_rows=1, expected_col_lines=8, first_col="s_no",
        fields={"s_no": I("S.No."), "item_code": S("ITEM CODE"), "item_name": S("ITEM NAME"), "unit": S("UNIT"),
                "rejection_qty": I("REJECTION QTY."), "vendor_name": S("VENDOR. NAME"),
                "reason_remarks": S("REASON / REMARKS")},
        required=["item_code", "rejection_qty"],
        rules=[("item_code looks like WLD602503", ["item_code"],
                lambda r: re.fullmatch(r"WLD\d{6}", r["item_code"].replace(" ", "").upper()) is not None)]),
}

def detect_template(pdf_path):
    name = pdf_path.stem.upper()
    for key, t in TEMPLATES.items():
        if any(m in name for m in t["match"]):
            return key
    return None

for p in PDFS:
    print(f"{detect_template(p) or '— no template —':32s} ← {p.name}")

## 4. Preprocessing
Each step writes an image to `<doc>/debug/` so any bad extraction can be traced back to its cause.

| Step | Why (seen on these scans) |
|---|---|
| Render page with its PDF transform | Scrap Note & Weld Shop scans are stored **upside down**; only the PDF placement matrix flips them |
| Normalise width to 2480 px | All pixel thresholds assume ~300 dpi A4 |
| Deskew on printed horizontal lines | Scans are tilted 0.2–1.0°, enough to smear a row line across two rows |
| Flatten lighting | Grey scanner/fold **shadow band** ~75 % across the page |
| Line detection by morphology + projection | Forms are fully ruled; long strokes = lines, handwriting disappears |
| Drop asymmetric lines | The shadow band edge looks like a column line (paper brighter on one side only) |
| Ink per row (lines removed) | Finds the few filled rows; empty rows → 0 ink, reviewer's slash < 1000 |
| Group rows | An entry often wraps onto the next ruled line ("Pipe 12.70x / 1.20x498") |

In [ ]:
class Debug:
    """Saves numbered debug images for one document."""
    def __init__(self, root):
        self.root = Path(root); self.root.mkdir(parents=True, exist_ok=True); self.n = 0
    def save(self, name, img):
        path = self.root / f"{self.n:02d}_{name}.png"; self.n += 1
        cv2.imwrite(str(path), img)
        return path

def show(img, title="", width=16):
    h, w = img.shape[:2]
    plt.figure(figsize=(width, width * h / w))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap="gray")
    plt.title(title); plt.axis("off"); plt.show()

def load_page(pdf_path):
    # Render at the scan's native resolution. Rendering (instead of extracting the embedded JPEG)
    # applies the PDF's placement matrix, which is what un-flips upside-down scans.
    page = pymupdf.open(pdf_path)[0]
    infos = page.get_image_info()
    if infos:
        info = max(infos, key=lambda i: i["width"] * i["height"])
        zoom = info["width"] / max(info["bbox"][2] - info["bbox"][0], 1)
    else:
        zoom = 300 / 72                      # digital PDF without a scan: 300 dpi
    pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), colorspace=pymupdf.csRGB, alpha=False)
    rgb = np.frombuffer(pix.samples, np.uint8).reshape(pix.h, pix.stride)[:, :pix.w * 3].reshape(pix.h, pix.w, 3)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), dict(zoom=round(zoom, 3), scanned=bool(infos))

def normalise_width(img, width=TARGET_WIDTH):
    s = width / img.shape[1]
    return img if abs(s - 1) < 0.01 else cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)

def to_gray(img): return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def binarize(gray):  # ink = white
    return cv2.adaptiveThreshold(~gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 15, -2)

def horizontal_lines(bw):
    return cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (bw.shape[1] // 25, 1)))

def vertical_lines(bw):
    return cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, bw.shape[0] // 40)))

def deskew(img):
    gray = to_gray(img)
    h = horizontal_lines(binarize(gray))
    segs = cv2.HoughLinesP(h, 1, np.pi / 1800, 200, minLineLength=gray.shape[1] // 3, maxLineGap=20)
    if segs is None:
        return img, 0.0
    # HoughLinesP returns (N, 1, 4) on most OpenCV builds but (N, 4) on some - reshape handles both
    angles = [np.degrees(np.arctan2(y2 - y1, x2 - x1)) for x1, y1, x2, y2 in np.asarray(segs).reshape(-1, 4)]
    angles = [a for a in angles if abs(a) < 5]
    angle = float(np.median(angles)) if angles else 0.0
    M = cv2.getRotationMatrix2D((img.shape[1] / 2, img.shape[0] / 2), angle, 1)
    return cv2.warpAffine(img, M, img.shape[1::-1], flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255)), angle

def flatten_lighting(img):
    bg = cv2.morphologyEx(img, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (51, 51)))
    return cv2.divide(img, bg, scale=255)

def line_positions(mask, axis, frac):
    prof = mask.sum(axis=axis) / 255
    if prof.max() == 0:
        return []
    idx = np.where(prof > frac * prof.max())[0]
    groups = np.split(idx, np.where(np.diff(idx) > 5)[0] + 1)
    return [int(g.mean()) for g in groups if len(g)]

def filter_columns(gray, rows, cands):
    # a printed line is dark with equally bright paper on both sides; a shadow edge is bright on one side only
    kept, report = [], []
    strip = gray[rows[0]:rows[-1]]
    for k, x in enumerate(cands):
        line  = strip[:, max(x - 2, 0):x + 3].min(axis=1).mean()
        left  = strip[:, max(x - 15, 0):max(x - 8, 1)].mean()
        right = strip[:, x + 8:x + 15].mean() if x + 15 <= strip.shape[1] else 255.0
        border = k in (0, len(cands) - 1)
        ok = (min(left, right) - line > 25) and (border or abs(left - right) < 12)
        report.append(dict(x=x, line=round(line), left=round(left), right=round(right), kept=ok))
        if ok:
            kept.append(x)
    return kept, report

def ink_mask(gray, h, v):
    ink = ((gray < 160) * 255).astype(np.uint8)
    ink = cv2.subtract(ink, cv2.dilate(h | v, np.ones((7, 7), np.uint8)))
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink)
    return (np.isin(lab, np.where(st[:, 4] >= 25)[0][1:]) * 255).astype(np.uint8)

def with_margin(img, m=IMAGE_MARGIN):
    return cv2.copyMakeBorder(img, m, m, m, m, cv2.BORDER_CONSTANT, value=(255, 255, 255))

In [ ]:
def preprocess(pdf_path, tpl, doc_dir):
    dbg = Debug(doc_dir / "debug")
    inputs_dir = doc_dir / "model_inputs"; inputs_dir.mkdir(parents=True, exist_ok=True)
    warnings = []

    img, info = load_page(pdf_path);              dbg.save("rendered", img)
    img = normalise_width(img)
    img, angle = deskew(img);                     dbg.save("deskewed", img)
    raw_gray = to_gray(img)                                       # before flattening: used to judge line contrast
    img = flatten_lighting(img);                  dbg.save("flattened", img)
    gray = to_gray(img)
    bw = binarize(gray);                          dbg.save("binary", bw)
    h, v = horizontal_lines(bw), vertical_lines(bw)
    dbg.save("horizontal_lines", h);              dbg.save("vertical_lines", v)

    rows = line_positions(h, 1, 0.5)
    cands = line_positions(v, 0, 0.3)
    if len(rows) < tpl["header_rows"] + tpl["footer_rows"] + 2 or len(cands) < 2:
        raise RuntimeError(f"table grid not found (rows={len(rows)}, column lines={len(cands)})")
    cols, col_report = filter_columns(raw_gray, rows, cands)
    if len(cols) != tpl["expected_col_lines"]:
        warnings.append(f"found {len(cols)} column lines, template expects {tpl['expected_col_lines']}")

    ink = ink_mask(gray, h, v);                   dbg.save("ink_no_lines", ink)
    x0, x1 = cols[0], cols[-1]
    hr = tpl["header_rows"]
    data_rows = range(hr, len(rows) - 1 - tpl["footer_rows"])
    row_ink = {i: int(ink[rows[i] + 4:rows[i + 1] - 4, x0:x1].sum() / 255) for i in data_rows}
    filled = [i for i, s in row_ink.items() if s > MIN_ROW_INK]

    def first_col_ink(i):
        return ink[rows[i] + 4:rows[i + 1] - 4, cols[0] + 4:cols[1] - 4].sum() / 255
    records = []
    for i in filled:
        if records and i == records[-1][-1] + 1 and first_col_ink(i) < CONTINUATION_MAX_INK:
            records[-1].append(i)          # continuation of the previous entry
        else:
            records.append([i])
    if not records:
        warnings.append("no handwritten rows found")

    # overlay: rows red, kept columns blue, dropped columns yellow, records green
    ov = img.copy()
    for i, y in enumerate(rows):
        cv2.line(ov, (0, y), (ov.shape[1], y), (0, 0, 255), 2)
        cv2.putText(ov, f"r{i}", (5, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    for c in col_report:
        color = (255, 0, 0) if c["kept"] else (0, 200, 255)
        cv2.line(ov, (c["x"], 0), (c["x"], ov.shape[0]), color, 2 if c["kept"] else 5)
    for k, rec in enumerate(records):
        cv2.rectangle(ov, (x0, rows[rec[0]]), (x1, rows[rec[-1] + 1]), (0, 170, 0), 6)
        cv2.putText(ov, f"record {k}", (x0 + 10, rows[rec[0]] + 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 170, 0), 3)
    overlay_path = dbg.save("grid_overlay", ov)

    # model inputs: the exact files that will be sent to Qwen
    xa, xb = max(x0 - SIDE_PAD, 0), min(x1 + SIDE_PAD, img.shape[1])
    title = with_margin(img[max(rows[0] - TOP_PAD, 0):rows[hr - 1] + BOTTOM_PAD, xa:xb])   # month is written across the line
    title_path = inputs_dir / "title.png"; cv2.imwrite(str(title_path), title)
    header = img[rows[hr - 1]:rows[hr], xa:xb]
    rec_items = []
    for k, rec in enumerate(records):
        # top: skip padding if the row above is the header or another entry (its descenders live there)
        top_pad = 0 if (rec[0] == hr or (rec[0] - 1) in filled) else TOP_PAD
        body = img[rows[rec[0]] - top_pad:min(rows[rec[-1] + 1] + BOTTOM_PAD, img.shape[0]), xa:xb]
        crop = with_margin(np.vstack([header, body]))
        path = inputs_dir / f"record_{k:02d}.png"; cv2.imwrite(str(path), crop)
        rec_items.append(dict(idx=k, physical_rows=rec, image=path, size_wh=(crop.shape[1], crop.shape[0])))

    return dict(pdf=pdf_path.name, template=tpl, angle=round(angle, 2), render=info, rows=rows, cols=cols,
                col_report=col_report, row_ink=row_ink, filled=filled, records=rec_items,
                title_image=title_path, overlay=overlay_path, warnings=warnings)

In [ ]:
DOCS = []
for pdf in PDFS:
    key = detect_template(pdf)
    if key is None:
        print(f"SKIP {pdf.name}: no matching template"); continue
    doc_dir = OUTPUT_DIR / re.sub(r"[^A-Za-z0-9]+", "_", pdf.stem).strip("_")
    if doc_dir.exists():
        shutil.rmtree(doc_dir)
    try:
        d = preprocess(pdf, TEMPLATES[key], doc_dir)
    except Exception as e:
        import traceback
        print(f"FAIL {pdf.name}: {type(e).__name__}: {e}")
        traceback.print_exc(limit=-3)          # last frames: shows the failing line
        continue
    d.update(key=key, doc_dir=doc_dir)
    DOCS.append(d)
    dropped = [c["x"] for c in d["col_report"] if not c["kept"]]
    print(f"{pdf.name:42s} {key:32s} deskew={d['angle']:+.2f}°  rows={len(d['rows'])}  "
          f"cols={len(d['cols'])}  dropped_cols={dropped}  records={[r['physical_rows'] for r in d['records']]}")
    for w in d["warnings"]:
        print("    ⚠", w)

pd.DataFrame([dict(pdf=d["pdf"], template=d["key"], deskew_deg=d["angle"], row_lines=len(d["rows"]),
                   col_lines=len(d["cols"]), records=len(d["records"]), warnings="; ".join(d["warnings"]))
              for d in DOCS]).to_csv(OUTPUT_DIR / "preprocess_summary.csv", index=False)

### Look at the grid overlays and the exact model inputs

In [ ]:
for d in DOCS:
    ov = cv2.imread(str(d["overlay"]))
    y1 = min(ov.shape[0], d["rows"][-1] + 100)
    show(ov[:y1], f"{d['pdf']}  – red rows, blue columns, yellow = dropped fake line, green = records", width=14)
    for r in d["records"]:
        show(cv2.imread(str(r["image"])), f"model input: {r['image'].name}  {r['size_wh'][0]}x{r['size_wh'][1]} px", width=14)

## 5. Load Qwen3-VL-4B
fp16 on T4 (no bf16 support), bf16 on L4/A100. First load downloads ~9 GB.

In [ ]:
model = processor = None
if not DRY_RUN:
    from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype, device_map="auto")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model.eval()
    print("loaded", MODEL_ID, dtype, f"| GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
else:
    print("DRY_RUN: model not loaded")

## 6. Prompts and model call
One call per **record crop** (column-header strip + that entry's rows) and one call for the **title band** (form title + month).
The prompt forbids calculating or correcting values, so the validation rules in step 7 stay meaningful.

In [ ]:
def record_prompt(tpl):
    lines = [
        f'This image is cut from a handwritten factory QA register: "{tpl["title"]}" (Meenakshi Polymers, Haridwar).',
        "The top strip shows the printed column headers. Below it is ONE handwritten entry. "
        "The entry may continue onto a second ruled line - treat those lines as the same entry.",
        "Extract the entry into JSON with exactly these keys:",
    ]
    for key, (header, typ) in tpl["fields"].items():
        lines.append(f'- "{key}": {typ} or null   (printed column: "{header}")')
    lines += [
        "Rules:",
        "- Copy exactly what is handwritten. Do NOT calculate, correct, or infer any value.",
        "- Empty or illegible cell -> null.",
        '- Integer fields: digits only, e.g. "07" -> 7, "12,274" -> 12274.',
        '- In text fields, a ditto mark meaning "same as above" (", 〃, or two short strokes like 11) -> "DITTO".',
        "- Keep spelling as written. Join text wrapped onto the next line with a single space.",
        "- Dates: copy as written, e.g. 03/6/26.",
        'Return only JSON in this form: {"entries": [ { ...keys above... } ]}',
    ]
    return "\n".join(lines)

TITLE_PROMPT = (
    "This is the top of a scanned factory form. Read the printed form title and the handwritten month "
    '(written after "MONTH" or "Month-", may be blank). Copy the month exactly as written. '
    'Return only JSON: {"form_title": string, "month": string or null}'
)

def parse_json(text):
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    m = re.search(r"\{.*\}", t, re.S)
    if not m:
        return None, "no JSON object in output"
    try:
        return json.loads(m.group(0)), None
    except json.JSONDecodeError as e:
        return None, f"invalid JSON: {e}"

def qwen_generate(image_path, prompt):
    if DRY_RUN:
        return dict(raw="", seconds=0.0, input_tokens=None, output_tokens=None, grid_thw=None, model_view_wh=None)
    messages = [{"role": "user", "content": [{"type": "image", "image": str(image_path)},
                                             {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new = out[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(new, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    grid = [int(x) for x in inputs["image_grid_thw"][0].tolist()]          # [t, h_patches, w_patches]
    patch = getattr(processor.image_processor, "patch_size", 16)
    return dict(raw=text, seconds=round(time.time() - t0, 2), input_tokens=int(inputs["input_ids"].shape[1]),
                output_tokens=int(new.shape[1]), grid_thw=grid, model_view_wh=(grid[2] * patch, grid[1] * patch))

def save_model_view(image_path, wh, out_dir):
    # the image resized to the resolution Qwen actually processed (from image_grid_thw)
    if not wh:
        return None
    out_dir.mkdir(parents=True, exist_ok=True)
    p = out_dir / Path(image_path).name
    cv2.imwrite(str(p), cv2.resize(cv2.imread(str(image_path)), tuple(wh), interpolation=cv2.INTER_AREA))
    return p

print(record_prompt(TEMPLATES["SUPPLIER_SEGREGATION"]))

In [ ]:
def log_io(doc_dir, entry):
    with open(doc_dir / "model_io.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False, default=str) + "\n")

for d in DOCS:
    tpl, doc_dir = d["template"], d["doc_dir"]
    (doc_dir / "model_io.jsonl").unlink(missing_ok=True)

    r = qwen_generate(d["title_image"], TITLE_PROMPT)
    parsed, err = parse_json(r["raw"]) if r["raw"] else (None, "dry run")
    d["title"] = dict(**r, parsed=parsed, error=err,
                      model_view=save_model_view(d["title_image"], r["model_view_wh"], doc_dir / "model_view"))
    log_io(doc_dir, dict(kind="title", image=d["title_image"], prompt=TITLE_PROMPT, **d["title"]))

    prompt = record_prompt(tpl)
    for rec in d["records"]:
        r = qwen_generate(rec["image"], prompt)
        parsed, err = parse_json(r["raw"]) if r["raw"] else (None, "dry run")
        rec.update(prompt=prompt, **r, parsed=parsed, error=err,
                   model_view=save_model_view(rec["image"], r["model_view_wh"], doc_dir / "model_view"))
        log_io(doc_dir, dict(kind="record", idx=rec["idx"], physical_rows=rec["physical_rows"], image=rec["image"],
                             image_size_wh=rec["size_wh"], prompt=prompt, **{k: rec[k] for k in
                             ("raw", "seconds", "input_tokens", "output_tokens", "grid_thw", "model_view_wh",
                              "parsed", "error", "model_view")}))
        print(f"{d['pdf'][:34]:34s} record {rec['idx']}  {rec['seconds']:5.1f}s  "
              f"view={rec['model_view_wh']}  tokens_in={rec['input_tokens']}  {err or 'ok'}")

## 7. Validate
Per record: types, required fields, ditto marks resolved from the previous entry, arithmetic rules, code formats,
date format, and "exactly one entry per crop". Anything that fails → `status = review`.

In [ ]:
DATE_RE = re.compile(r"^\s*\d{1,2}\s*[/.\-]\s*\d{1,2}\s*[/.\-]\s*\d{2,4}\s*$")

def to_int(v):
    if v is None or isinstance(v, bool): return None
    if isinstance(v, int): return v
    if isinstance(v, float): return int(v) if v.is_integer() else None
    s = re.sub(r"[,\s]", "", str(v))
    return int(s) if re.fullmatch(r"-?\d+", s) else None

def validate_doc(d):
    tpl, rows_out, prev = d["template"], [], {}
    month = (d.get("title", {}).get("parsed") or {}).get("month")
    for rec in d["records"]:
        issues, info = [], []
        entries = (rec.get("parsed") or {}).get("entries")
        if rec.get("error"):
            issues.append(rec["error"])
        if entries is not None and len(entries) != 1:
            issues.append(f"model returned {len(entries)} entries for one crop")
        e = entries[0] if entries else {}
        out = {}
        for key, (_, typ) in tpl["fields"].items():
            val = e.get(key)
            if typ == "integer":
                iv = to_int(val)
                if val not in (None, "") and iv is None:
                    issues.append(f"{key}: not a number ({val!r})")
                out[key] = iv
            else:
                s = None if val is None else str(val).strip() or None
                if s is not None and s.upper() == "DITTO":
                    s = prev.get(key)
                    info.append(f"{key}: ditto → {s!r}")
                out[key] = s
        for key in tpl["required"]:
            if out.get(key) is None:
                issues.append(f"missing {key}")
        for desc, needed, fn in tpl["rules"]:
            if any(out.get(k) is None for k in needed):
                continue                                  # already reported as missing
            try:
                if not fn(out):
                    issues.append(f"rule failed: {desc}")
            except Exception as ex:
                issues.append(f"rule error: {desc} ({ex})")
        if out.get("date") and not DATE_RE.match(out["date"]):
            issues.append(f"date format: {out['date']!r}")
        prev = {k: v for k, v in out.items() if v is not None} | {k: v for k, v in prev.items() if out.get(k) is None}
        status = "not_run" if DRY_RUN else ("auto_ok" if not issues else "review")
        rec.update(values=out, issues=issues, info=info, status=status)
        rows_out.append(dict(doc=d["pdf"], form=d["key"], month=month, record_idx=rec["idx"],
                             physical_rows=",".join(map(str, rec["physical_rows"])), **out,
                             status=status, issues="; ".join(issues), notes="; ".join(info),
                             input_image=str(rec["image"])))
    return rows_out

ALL_ROWS = [r for d in DOCS for r in validate_doc(d)]
results = pd.DataFrame(ALL_ROWS)
results.to_csv(OUTPUT_DIR / "results.csv", index=False, encoding="utf-8-sig")
with pd.ExcelWriter(OUTPUT_DIR / "results.xlsx") as xw:
    for key in results["form"].unique() if len(results) else []:
        sub = results[results["form"] == key].dropna(axis=1, how="all")
        sub.to_excel(xw, sheet_name=key[:31], index=False)
    if len(results):
        results[results["status"] != "auto_ok"][["doc", "record_idx", "status", "issues", "input_image"]] \
            .to_excel(xw, sheet_name="needs_review", index=False)
print(results["status"].value_counts().to_dict() if len(results) else "no records")
results.drop(columns=["input_image"], errors="ignore")

## 8. Debug report
`report.html`: for every record the **exact image sent**, the resolution Qwen processed, the raw output,
the parsed values and the validation issues side by side. Images are linked (relative paths), so open it with the
folder next to it: in Drive, download `qwen3vl_output.zip` (step 10), unzip, open `report.html`.
For a quick look without leaving Colab, use `inspect()` below.

In [ ]:
def rel(path):
    # images are linked, not embedded: keep report.html next to the doc folders (e.g. the unzipped Drive zip)
    return Path(os.path.relpath(path, OUTPUT_DIR)).as_posix()

def esc(x): return html.escape("" if x is None else str(x))

def build_report(docs):
    css = ("body{font-family:system-ui,Segoe UI,sans-serif;margin:24px;background:#fafafa;color:#222}"
           "h2{margin-top:40px;border-bottom:2px solid #ccc}.rec{background:#fff;border:1px solid #ddd;border-radius:8px;"
           "padding:12px;margin:14px 0}.ok{border-left:6px solid #2e7d32}.review{border-left:6px solid #c62828}"
           ".not_run{border-left:6px solid #999}img{max-width:100%;border:1px solid #bbb}table{border-collapse:collapse}"
           "td,th{border:1px solid #ddd;padding:3px 8px;font-size:13px;text-align:left}pre{background:#f3f3f3;padding:8px;"
           "white-space:pre-wrap;font-size:12px}.meta{color:#666;font-size:12px}.issue{color:#c62828}.note{color:#1565c0}")
    out = [f"<html><head><meta charset='utf-8'><title>Qwen extraction report</title><style>{css}</style></head><body>",
           f"<h1>Qwen3-VL extraction report</h1><p class='meta'>model: {esc(MODEL_ID)} · dry run: {DRY_RUN} · "
           f"{time.strftime('%Y-%m-%d %H:%M')}</p>"]
    for d in docs:
        t = d.get("title", {})
        out.append(f"<h2>{esc(d['pdf'])}</h2><p class='meta'>template {esc(d['key'])} · deskew {d['angle']}° · "
                   f"{len(d['rows'])} row lines · {len(d['cols'])} column lines · {len(d['records'])} records · "
                   f"month: <b>{esc((t.get('parsed') or {}).get('month'))}</b></p>")
        for w in d["warnings"]:
            out.append(f"<p class='issue'>⚠ {esc(w)}</p>")
        out.append(f"<details><summary>grid overlay</summary><img src='{rel(d['overlay'])}'></details>")
        out.append(f"<details><summary>title band sent to model</summary><img src='{rel(d['title_image'])}'>"
                   f"<pre>{esc(t.get('raw'))}</pre></details>")
        for rec in d["records"]:
            cls = {"auto_ok": "ok"}.get(rec.get("status"), rec.get("status", "not_run"))
            vals = "".join(f"<tr><th>{esc(k)}</th><td>{esc(v)}</td></tr>" for k, v in (rec.get("values") or {}).items())
            issues = "".join(f"<div class='issue'>✗ {esc(i)}</div>" for i in rec.get("issues", []))
            notes = "".join(f"<div class='note'>• {esc(i)}</div>" for i in rec.get("info", []))
            out.append(
                f"<div class='rec {cls}'><b>record {rec['idx']}</b> · physical rows {rec['physical_rows']} · "
                f"status <b>{esc(rec.get('status'))}</b>"
                f"<p class='meta'>sent: {esc(rec['image'].name)} {rec['size_wh'][0]}×{rec['size_wh'][1]} px · "
                f"processed by model at: {esc(rec.get('model_view_wh'))} · grid_thw {esc(rec.get('grid_thw'))} · "
                f"tokens in/out {esc(rec.get('input_tokens'))}/{esc(rec.get('output_tokens'))} · {esc(rec.get('seconds'))} s</p>"
                f"<img src='{rel(rec['image'])}'>"
                + (f"<details><summary>as processed by the model ({esc(rec.get('model_view_wh'))})</summary>"
                   f"<img src='{rel(rec['model_view'])}'></details>" if rec.get("model_view") else "") +
                f"<div style='display:flex;gap:16px;margin-top:8px'><div><table>{vals}</table>{issues}{notes}</div>"
                f"<div style='flex:1'><details><summary>raw model output</summary><pre>{esc(rec.get('raw'))}</pre></details>"
                f"<details><summary>prompt</summary><pre>{esc(rec.get('prompt', record_prompt(d['template'])))}</pre></details>"
                f"</div></div></div>")
    out.append("</body></html>")
    return "\n".join(out)

report_path = OUTPUT_DIR / "report.html"
report_path.write_text(build_report(DOCS), encoding="utf-8")
print("report:", report_path.resolve(), f"({report_path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# quick look at one record inside the notebook
def inspect(doc_index=0, record_index=0):
    d = DOCS[doc_index]; rec = d["records"][record_index]
    show(cv2.imread(str(rec["image"])), f"{d['pdf']} · record {record_index} · sent {rec['size_wh']} · model saw {rec.get('model_view_wh')}")
    print("raw output:\n", rec.get("raw"))
    print("\nvalues:", json.dumps(rec.get("values"), indent=1, ensure_ascii=False))
    print("issues:", rec.get("issues"), "| notes:", rec.get("info"))

if DOCS and DOCS[0]["records"]:
    inspect(0, 0)

## 9. Accuracy against ground truth
1. `truth_template.json` is written from the current predictions.
2. Open it from Drive (`qwen3vl_output/truth_template.json`), **correct every value by looking at the scan**,
   set `"verified": true`, save it as `ground_truth.json` in the same Drive folder (or in `Datasetpdf`) and re-run this cell.

Scoring: integers must match exactly; text is compared after normalising case/spaces, and CER (character error rate) is reported.

In [ ]:
truth_tpl = [dict(doc=r["doc"], record_idx=r["record_idx"], verified=False,
                  values={k: r.get(k) for k in TEMPLATES[r["form"]]["fields"]}) for r in ALL_ROWS]
(OUTPUT_DIR / "truth_template.json").write_text(json.dumps(truth_tpl, indent=1, ensure_ascii=False), encoding="utf-8")

def levenshtein(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def norm(s): return re.sub(r"\s+", " ", str(s)).strip().lower()

truth_file = next((p for p in (OUTPUT_DIR / "ground_truth.json", INPUT_DIR / "ground_truth.json") if p.exists()), None)
if truth_file is None or DRY_RUN:
    print("No ground_truth.json yet (or dry run) – wrote", OUTPUT_DIR / "truth_template.json")
else:
    truth = {(t["doc"], t["record_idx"]): t["values"] for t in json.loads(truth_file.read_text(encoding="utf-8"))}
    pred = {(r["doc"], r["record_idx"]): r for r in ALL_ROWS}
    rows = []
    for key, tv in truth.items():
        pv = pred.get(key, {})
        form = pv.get("form") or next((d["key"] for d in DOCS if d["pdf"] == key[0]), None)
        for field, want in tv.items():
            typ = TEMPLATES[form]["fields"][field][1] if form else "string"
            got = pv.get(field)
            if typ == "integer":
                ok, cer = (to_int(got) == to_int(want)), None
            else:
                a, b = norm(got or ""), norm(want or "")
                ok, cer = a == b, (levenshtein(a, b) / max(len(b), 1)) if (a or b) else 0.0
            rows.append(dict(doc=key[0], record_idx=key[1], field=field, type=typ, truth=want, pred=got,
                             correct=ok, cer=cer, record_found=key in pred))
    ev = pd.DataFrame(rows)
    ev.to_csv(OUTPUT_DIR / "evaluation.csv", index=False, encoding="utf-8-sig")
    print(f"records in truth: {len(truth)} · found by pipeline: {ev.groupby(['doc','record_idx'])['record_found'].first().mean():.0%}")
    print(ev.groupby("type").agg(fields=("correct", "size"), accuracy=("correct", "mean"), mean_cer=("cer", "mean")))
    print("\nper field:")
    print(ev.groupby("field")["correct"].mean().sort_values())
    print("\nerrors:")
    display(ev[~ev["correct"]][["doc", "record_idx", "field", "truth", "pred", "cer"]])

## 10. Zip the output (on Drive)
Everything is already on Drive in `qwen3vl_output/`. This also writes `MeenakshiPublic/qwen3vl_output.zip`
next to it - one file to download when you want to open `report.html` with its images on your computer.

In [ ]:
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)      # -> <DRIVE_ROOT>/qwen3vl_output.zip
print(f"zipped: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    print("Files are on Google Drive → MyDrive/MeenakshiPublic/qwen3vl_output (Drive may take a minute to sync)")